# MOD-08 · NB-08 — Capstone: Full Deployment Pipeline
### Health Analytics with Python · Module 08: Reproducible Research & Deployment
---
**This capstone integrates NB-01 through NB-07 into a complete production-ready clinical ML deployment.**

**Pipeline stages:**
1. Reproducible data provenance (checksums, MANIFEST.json)
2. Multi-model training with run logging
3. Automated model validation (unit tests + performance tests)
4. Fairness analysis across demographic subgroups
5. Model Card generation
6. FastAPI endpoint simulation
7. Publication-quality 8-panel summary figure (300 dpi)
8. Deployment readiness checklist

**Research context:** 30-day hospital readmission prediction — from raw data to production API

**Estimated time:** 4 hours | **Level:** Advanced

## 0. Setup & Synthetic Clinical Cohort

In [ ]:
import os, json, time, pickle
from pathlib import Path
from datetime import datetime
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              brier_score_loss, roc_curve, calibration_curve)
import warnings; warnings.filterwarnings("ignore")
os.makedirs("/tmp/mod08/capstone", exist_ok=True)
plt.rcParams.update({"figure.dpi":120,"savefig.dpi":300,"figure.facecolor":"white",
                     "axes.spines.top":False,"axes.spines.right":False})
SEED = 42; np.random.seed(SEED)

def sigmoid(x): return 1/(1+np.exp(-x))
N = 3000
age   = np.random.normal(60,15,N).clip(18,90).astype(int)
sex   = np.random.choice(["M","F"],N,p=[0.48,0.52])
race  = np.random.choice(["White","Black","Hispanic","Asian","Other"],N,p=[0.60,0.18,0.12,0.06,0.04])
payer = np.random.choice(["Medicare","Medicaid","Private","Self-pay"],N,p=[0.40,0.22,0.28,0.10])
los   = np.random.gamma(2.5,1.8,N).clip(1,30).astype(int)
dm    = np.random.binomial(1,0.35,N); hf = np.random.binomial(1,0.22,N)
copd  = np.random.binomial(1,0.18,N); smoking = np.random.binomial(1,0.25,N)
n_com = dm+hf+copd
logit = -3.2+0.6*hf+0.5*dm+0.3*copd+0.02*(age-60)+0.2*(los>7).astype(int)+np.random.normal(0,0.3,N)
readmitted = np.random.binomial(1,sigmoid(logit),N)
df = pd.DataFrame({"age":age,"sex":sex,"race":race,"payer":payer,
                   "los_days":los,"diabetes":dm,"hf":hf,"copd":copd,
                   "smoking":smoking,"n_comorb":n_com,"readmitted":readmitted})
FEATURES = ["age","los_days","diabetes","hf","copd","n_comorb"]
X_tr,X_te,y_tr,y_te = train_test_split(df[FEATURES],df["readmitted"],test_size=0.2,stratify=df["readmitted"],random_state=SEED)
df_tr = df.iloc[X_tr.index].reset_index(drop=True)
df_te = df.iloc[X_te.index].reset_index(drop=True)
print(f"Capstone dataset: N={N} | Readmission: {readmitted.mean()*100:.1f}% | Features: {len(FEATURES)}")


## 1. Training Run Log & Model Selection

In [ ]:
# Step 1: Train multiple models and track with simple registry
import hashlib

models_config = {
    "LR_baseline": LogisticRegression(C=1,max_iter=500,class_weight="balanced",random_state=SEED),
    "GBM_v1"     : GradientBoostingClassifier(n_estimators=200,max_depth=4,learning_rate=0.05,random_state=SEED),
    "GBM_v2_deep": GradientBoostingClassifier(n_estimators=300,max_depth=6,learning_rate=0.03,subsample=0.8,random_state=SEED),
}

run_log = []
fitted_models = {}
print("Model training run log:")
print(f"{'Run name':20s} {'AUC':>8s} {'AP':>8s} {'Brier':>8s} {'Time(s)':>9s}")
print("-"*55)

for name, m in models_config.items():
    t0 = time.time()
    m.fit(X_tr, y_tr)
    prob = m.predict_proba(X_te)[:,1]
    metrics = {
        "auc"  : roc_auc_score(y_te,prob),
        "ap"   : average_precision_score(y_te,prob),
        "brier": brier_score_loss(y_te,prob),
    }
    elapsed = time.time()-t0
    run_log.append({"run_name":name,"elapsed_s":round(elapsed,2),**metrics})
    fitted_models[name] = (m, prob)
    print(f"  {name:18s} {metrics['auc']:>8.4f} {metrics['ap']:>8.4f} {metrics['brier']:>8.4f} {elapsed:>9.2f}")

runs_df = pd.DataFrame(run_log)
best_name = runs_df.loc[runs_df.auc.idxmax(),"run_name"]
best_model, best_prob = fitted_models[best_name]
print(f"\nBest model: {best_name} (AUC={runs_df.auc.max():.4f})")

# Save best model
model_path = Path("/tmp/mod08/capstone/readmission_model_v1.pkl")
pickle.dump(best_model, open(model_path,"wb"))
print(f"Model saved: {model_path}")


## 2. Data Provenance & Fairness

In [ ]:
# Step 2: Data provenance and reproducibility
import hashlib

def sha256_file(path):
    h = hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(8192),b""): h.update(chunk)
    return h.hexdigest()

# Step 3: Fairness analysis
best_prob_arr = best_prob
fairness = []
auc_overall = roc_auc_score(y_te, best_prob_arr)
for col, groups in [("race",["White","Black","Hispanic","Asian"]),
                     ("payer",["Medicare","Medicaid","Private","Self-pay"]),
                     ("sex",["M","F"])]:
    for grp in groups:
        mask = df_te[col]==grp
        if mask.sum()<20: continue
        sub_y = y_te[mask.values]; sub_p = best_prob_arr[mask.values]
        if sub_y.nunique()<2: continue
        auc_g = roc_auc_score(sub_y,sub_p)
        fairness.append({"Group":f"{col}={grp}","N":mask.sum(),"AUC":round(auc_g,4),"Delta":round(auc_g-auc_overall,4)})

fairness_df = pd.DataFrame(fairness)
print(f"Overall AUC: {auc_overall:.4f}")
print("Fairness analysis (sorted by AUC delta):")
display(fairness_df.sort_values("Delta"))
max_disparity = fairness_df.Delta.abs().max()
print(f"Max disparity: {max_disparity:.4f} ({'PASS' if max_disparity<0.05 else 'REVIEW REQUIRED'})")


## 3. Automated Validation Tests

In [ ]:
# Step 4: Model validation tests
import unittest

class CapstoneModelTests(unittest.TestCase):
    def test_auc_above_minimum(self):
        self.assertGreater(auc_overall, 0.65, f"AUC {auc_overall:.4f} below minimum 0.65")

    def test_predictions_in_range(self):
        self.assertTrue((best_prob_arr>=0).all() and (best_prob_arr<=1).all())

    def test_direction_of_effect(self):
        X_high = pd.DataFrame([{"age":80,"los_days":14,"diabetes":1,"hf":1,"copd":1,"n_comorb":3}])
        X_low  = pd.DataFrame([{"age":25,"los_days":1,"diabetes":0,"hf":0,"copd":0,"n_comorb":0}])
        p_high = float(best_model.predict_proba(X_high)[:,1])
        p_low  = float(best_model.predict_proba(X_low)[:,1])
        self.assertGreater(p_high, p_low)

    def test_no_nan_predictions(self):
        self.assertFalse(np.isnan(best_prob_arr).any())

    def test_positive_rate_in_range(self):
        self.assertGreater(y_te.mean(), 0.05)
        self.assertLess(y_te.mean(), 0.50)

loader = unittest.TestLoader()
suite  = loader.loadTestsFromTestCase(CapstoneModelTests)
result = unittest.TextTestRunner(verbosity=2).run(suite)
tests_passed = result.testsRun - len(result.failures) - len(result.errors)
print(f"\nValidation: {tests_passed}/{result.testsRun} tests passed")


## 4. FastAPI Endpoint Simulation

In [ ]:
# Simulate the production API endpoint
import json

def simulate_api(age, los_days, diabetes, hf, copd, patient_id=None):
    X = pd.DataFrame([{"age":age,"los_days":los_days,"diabetes":diabetes,
                        "hf":hf,"copd":copd,"n_comorb":diabetes+hf+copd}])
    if not all(isinstance(v,(int,float)) for v in [age,los_days,diabetes,hf,copd]):
        return {"error":"422 Unprocessable Entity","detail":"Invalid field types"}
    if not (18<=age<=90): return {"error":"422","detail":f"age must be 18-90, got {age}"}
    if not (1<=los_days<=60): return {"error":"422","detail":f"los_days must be 1-60"}
    prob = float(best_model.predict_proba(X)[:,1])
    tier = ("Low" if prob<0.10 else "Moderate" if prob<0.20 else "High" if prob<0.35 else "Very High")
    return {"patient_id":patient_id,"readmission_probability":round(prob,4),
            "risk_tier":tier,"model_version":"1.0.0","timestamp":datetime.now().isoformat()}

test_patients = [
    (72,12,1,1,0,"PT-00142","High-risk HF+DM"),
    (35, 2,0,0,0,"PT-00289","Low-risk young"),
    (68, 8,1,0,1,"PT-00451","Moderate COPD+DM"),
    (150, 5,0,0,0,"PT-bad",  "Invalid age"),
]
print("POST /predict endpoint tests:")
print("-"*70)
for age,los,dm,hf,copd,pid,desc in test_patients:
    r = simulate_api(age,los,dm,hf,copd,patient_id=pid)
    if "error" in r:
        print(f"  [{desc:25s}] ERROR: {r['detail']}")
    else:
        print(f"  [{desc:25s}] {r['readmission_probability']:.4f} | {r['risk_tier']}")


## 5. Publication Summary Figure (300 dpi)

In [ ]:
# Step 5: Publication-quality summary figure (8 panels, 300 dpi)
fig = plt.figure(figsize=(22,16))
fig.suptitle(
    "End-to-End Deployment Pipeline: 30-Day Readmission Prediction Model\n"
    f"N={N:,} patients | Best model: {best_name} | AUC={auc_overall:.4f}",
    fontsize=14, y=1.01
)
gs = gridspec.GridSpec(3,4,figure=fig,hspace=0.48,wspace=0.38)

COLORS = ["#D65F5F","#4878CF","#6ACC65","#D4A017","#B47CC7"]

# A — Model comparison
ax_a = fig.add_subplot(gs[0,:2])
x = np.arange(len(runs_df)); w=0.28
for i,(metric,label,color) in enumerate([("auc","AUC-ROC","#4878CF"),("ap","Avg Precision","#D65F5F")]):
    bars = ax_a.bar(x+i*w-w/2, runs_df[metric], w, label=label, color=color, alpha=0.85, edgecolor="white")
    for bar,val in zip(bars,runs_df[metric]):
        ax_a.text(bar.get_x()+bar.get_width()/2,val+0.005,f"{val:.3f}",ha="center",fontsize=8)
ax_a.set_xticks(x); ax_a.set_xticklabels(runs_df.run_name, rotation=10, fontsize=9)
ax_a.set_ylabel("Score"); ax_a.set_title("A  Model Comparison (Training Run Log)",fontweight="bold")
ax_a.legend(fontsize=9); ax_a.set_ylim(0,1)

# B — ROC curve
ax_b = fig.add_subplot(gs[0,2])
fpr,tpr,_ = roc_curve(y_te,best_prob_arr)
ax_b.plot(fpr,tpr,color="#D65F5F",lw=2.5,label=f"AUC={auc_overall:.3f}")
ax_b.plot([0,1],[0,1],"k--",lw=1); ax_b.fill_between(fpr,tpr,alpha=0.1,color="#D65F5F")
ax_b.set_xlabel("1-Specificity"); ax_b.set_ylabel("Sensitivity")
ax_b.set_title("B  ROC Curve",fontweight="bold"); ax_b.legend(fontsize=9)

# C — Calibration
ax_c = fig.add_subplot(gs[0,3])
fp,mp = calibration_curve(y_te,best_prob_arr,n_bins=10)
ax_c.plot(mp,fp,"o-",color="#4878CF",lw=2,ms=6,label=f"Brier={brier_score_loss(y_te,best_prob_arr):.3f}")
ax_c.plot([0,1],[0,1],"k--",lw=1)
ax_c.set_xlabel("Predicted"); ax_c.set_ylabel("Observed fraction")
ax_c.set_title("C  Calibration Curve",fontweight="bold"); ax_c.legend(fontsize=9)

# D — Fairness bar chart
ax_d = fig.add_subplot(gs[1,:2])
fair_sorted = fairness_df.sort_values("AUC")
colors_fair = ["#D65F5F" if abs(d)>0.05 else "#4878CF" for d in fair_sorted.Delta]
bars_d = ax_d.bar(range(len(fair_sorted)), fair_sorted.AUC, color=colors_fair, alpha=0.85, edgecolor="white")
ax_d.axhline(auc_overall, color="black", ls="--", lw=1.5, label=f"Overall AUC={auc_overall:.3f}")
ax_d.axhline(auc_overall-0.05, color="orange", ls=":", lw=1.5, label="Disparity threshold")
ax_d.set_xticks(range(len(fair_sorted))); ax_d.set_xticklabels(fair_sorted.Group, rotation=35, fontsize=7)
ax_d.set_ylabel("AUC-ROC"); ax_d.set_title("D  Fairness Analysis by Subgroup",fontweight="bold")
ax_d.legend(fontsize=8); ax_d.set_ylim(0.6,0.9)

# E — Risk tier distribution
ax_e = fig.add_subplot(gs[1,2])
tiers = pd.cut(best_prob_arr,[0,0.10,0.20,0.35,1.01],labels=["Low","Moderate","High","Very High"])
tier_counts = tiers.value_counts().sort_index()
tier_colors = ["#4878CF","#6ACC65","#D4A017","#D65F5F"]
bars_e = ax_e.bar(tier_counts.index,tier_counts.values,color=tier_colors,alpha=0.85,edgecolor="white")
for bar,val in zip(bars_e,tier_counts.values):
    pct = val/len(best_prob_arr)*100
    ax_e.text(bar.get_x()+bar.get_width()/2,val+5,f"{pct:.0f}%",ha="center",fontsize=9)
ax_e.set_ylabel("N patients"); ax_e.set_title("E  Risk Tier Distribution",fontweight="bold")

# F — Readmission rate by tier
ax_f = fig.add_subplot(gs[1,3])
df_te["risk_tier"] = pd.cut(best_prob_arr,[0,0.10,0.20,0.35,1.01],labels=["Low","Moderate","High","Very High"])
tier_readmit = df_te.groupby("risk_tier",observed=True)["readmitted"].mean()*100
ax_f.bar(tier_readmit.index,tier_readmit.values,color=tier_colors,alpha=0.85,edgecolor="white")
ax_f.set_ylabel("Actual readmission rate (%)"); ax_f.set_title("F  Actual Rate by Predicted Tier",fontweight="bold")
for i,(t,r) in enumerate(tier_readmit.items()):
    ax_f.text(i,r+0.5,f"{r:.1f}%",ha="center",fontsize=9)

# G — Governance checklist progress
ax_g = fig.add_subplot(gs[2,:2])
governance_items = [
    ("Data MANIFEST.json with SHA-256", True),
    ("requirements.txt (pinned)", True),
    ("Pipeline.log recorded", True),
    ("Unit tests passing", tests_passed==result.testsRun),
    ("Fairness analysis complete", True),
    ("Model Card written", True),
    ("CI/CD workflow defined", True),
    ("FastAPI endpoint tested", True),
]
y_pos = range(len(governance_items))
for i,(item,done) in enumerate(governance_items):
    color = "#6ACC65" if done else "#D65F5F"
    label = "[x]" if done else "[ ]"
    ax_g.barh(i,1,color=color,alpha=0.7,edgecolor="white")
    ax_g.text(0.02,i,f"{label} {item}",va="center",fontsize=9)
ax_g.set_yticks([]); ax_g.set_xticks([]); ax_g.set_xlim(0,1.3)
done_count = sum(1 for _,d in governance_items if d)
ax_g.set_title(f"G  Deployment Checklist ({done_count}/{len(governance_items)} complete)",fontweight="bold")

# H — Deployment architecture
ax_h = fig.add_subplot(gs[2,2:])
ax_h.axis("off")
arch_text = (
    "H  Production Deployment Architecture\n\n"
    "EHR System\n"
    "    |\n"
    "    v\n"
    "FastAPI (4 workers)   <-- POST /predict\n"
    "    |\n"
    "    v\n"
    "GBM Model (pkl)       <-- models/v1.pkl\n"
    "    |\n"
    "    v\n"
    "JSON Response         <-- prob, tier, timestamp\n"
    "    |\n"
    "    +-> MLflow Tracking  (log each prediction)\n"
    "    +-> Audit Log        (patient_id, timestamp)\n"
    "    +-> Drift Monitor    (weekly AUC batch job)\n"
)
ax_h.text(0.05,0.95,arch_text,transform=ax_h.transAxes,fontsize=10,
           va="top",ha="left",family="monospace",
           bbox=dict(boxstyle="round",facecolor="#f8f9fa",alpha=0.8))
ax_h.set_title("H  Architecture",fontweight="bold")

plt.savefig("/tmp/mod08/capstone_deployment_summary.png",bbox_inches="tight",dpi=300)
plt.show()
print("Saved: capstone_deployment_summary.png (300 dpi)")


## 6. Deployment Narrative

In [ ]:
# Step 6: Methods narrative and deployment summary
DEPLOYMENT_NARRATIVE = (
    "METHODS NARRATIVE\n"
    "="*60 + "\n\n"
    "We developed and validated a 30-day hospital readmission prediction model\n"
    f"using a cohort of N={N:,} general medical-surgical patients (2019-2023).\n\n"
    "FEATURE ENGINEERING: Six pre-discharge features were selected based on\n"
    "clinical plausibility and availability in the EHR at discharge:\n"
    "age, length of stay, diabetes, heart failure, COPD, and comorbidity count.\n\n"
    "MODEL TRAINING: Three models were evaluated: logistic regression (baseline),\n"
    "gradient boosting v1, and gradient boosting v2 with deeper trees. Models were\n"
    "compared on AUC-ROC, Average Precision, and Brier score on a 20% holdout test set.\n\n"
    f"PERFORMANCE: Best model ({best_name}) achieved AUC-ROC={auc_overall:.4f} and\n"
    f"Average Precision={average_precision_score(y_te,best_prob_arr):.4f}. Calibration\n"
    f"was assessed with a calibration curve (Brier score={brier_score_loss(y_te,best_prob_arr):.4f}).\n\n"
    "FAIRNESS: Performance was evaluated across sex, race/ethnicity, and payer subgroups.\n"
    f"Maximum AUC disparity was {fairness_df.Delta.abs().max():.4f} (threshold: 0.05). "
    + ("No significant disparities detected.\n\n" if fairness_df.Delta.abs().max()<0.05 else "REVIEW REQUIRED.\n\n") +
    "DEPLOYMENT: The model is served via a FastAPI REST endpoint with Pydantic validation.\n"
    "All predictions are logged to MLflow. Weekly AUC audits and quarterly fairness reviews\n"
    "are scheduled. A Model Card accompanies this deployment per Mitchell et al. (2019).\n\n"
    "LIMITATIONS: Single-site training data; SES/social determinants not included;\n"
    "prospective validation recommended before scaling to additional sites.\n\n"
    f"Analysis date: {datetime.now().strftime('%Y-%m-%d')} | Environment: Python 3.10 | Seed: {SEED}\n"
)

print(DEPLOYMENT_NARRATIVE)
narrative_path = Path("/tmp/mod08/capstone/DEPLOYMENT_NARRATIVE.txt")
narrative_path.write_text(DEPLOYMENT_NARRATIVE)
print(f"Saved: {narrative_path}")


## 7. Deployment Readiness Checklist

In [ ]:
# Final deployment checklist summary
deployment_status = {
    "Reproducibility"     : {"requirements.txt":True,"MANIFEST.json":True,"pipeline.log":True,"random_seed":True},
    "Model Quality"       : {"unit_tests_pass":tests_passed==result.testsRun,"auc_above_threshold":auc_overall>0.65,"calibration_acceptable":brier_score_loss(y_te,best_prob_arr)<0.20},
    "Fairness"            : {"max_disparity_below_005":fairness_df.Delta.abs().max()<0.05},
    "Documentation"       : {"model_card_written":True,"narrative_written":True},
    "Deployment_Ready"    : {"fastapi_app_defined":True,"dockerfile_written":True,"cicd_workflow_defined":True},
    "Governance"          : {"monitoring_plan":True,"patient_explanation":True},
}

print("FINAL DEPLOYMENT READINESS SUMMARY")
print("="*55)
all_passed = True
for category, checks in deployment_status.items():
    passed = sum(1 for v in checks.values() if v)
    total  = len(checks)
    status = "READY" if passed==total else "NEEDS WORK"
    all_passed = all_passed and (passed==total)
    print(f"  {category:25s}: {passed}/{total} [{status}]")
    for check, ok in checks.items():
        icon = "✓" if ok else "✗"
        print(f"    {icon} {check}")
print("="*55)
print(f"  OVERALL STATUS: {'DEPLOYMENT APPROVED' if all_passed else 'NOT YET READY'}")


## 8. Final Knowledge Check
1. Your capstone pipeline takes 8 minutes to run end-to-end. Which steps should be cached and which must always re-run?
2. The fairness analysis passes (max disparity 0.04) but a hospital administrator notices that Medicaid patients are predicted high-risk at twice the rate of Private patients. Is this a problem even if AUC is equal?
3. You add `n_comorb` as a new feature. What changes are required across: training, API schema, model card, tests, and documentation?
4. Six months post-deployment, AUC drops from 0.79 to 0.72. The data team says "nothing changed." How do you investigate systematically?
5. A clinician asks: "How do I know your model is not just encoding race?" How do you respond technically and ethically?


In [ ]:
# Exercise 5 — Race encoding audit
# Test: does removing race (not in features) change predictions?
# Test: are predictions correlated with race even without using it as a feature?

from scipy.stats import f_oneway

# 1. ANOVA: do predicted probabilities differ significantly by race?
race_groups = [best_prob_arr[df_te.race.values==r] for r in df_te.race.unique() if (df_te.race.values==r).sum()>10]
f_stat, p_anova = f_oneway(*race_groups)
print(f"ANOVA of predictions by race: F={f_stat:.3f}, p={p_anova:.4f}")
if p_anova < 0.05:
    print("Predictions DO differ by race (even without using race as a feature)")
    print("  -> Investigate: which features mediate race? (e.g., payer type, comorbidities)")
    print("  -> Does this reflect real disparities or encode historical bias?")
else:
    print("No significant difference in predictions by race")

# 2. Check if adding race as a feature changes predictions
df_te_aug = df_te.copy()
df_te_aug["race_encoded"] = pd.Categorical(df_te.race).codes
X_with_race = df_te_aug[FEATURES + ["race_encoded"]]
model_with_race = GradientBoostingClassifier(n_estimators=100,random_state=SEED)
X_tr_aug = df_tr.copy(); X_tr_aug["race_encoded"] = pd.Categorical(df_tr.race).codes
model_with_race.fit(X_tr_aug[FEATURES+["race_encoded"]], y_tr)
prob_with_race = model_with_race.predict_proba(X_with_race)[:,1]
corr_change = np.corrcoef(best_prob_arr, prob_with_race)[0,1]
print(f"\nCorrelation of predictions (without race vs with race): {corr_change:.4f}")
print(f"{'Models nearly identical (race adds little)' if corr_change>0.97 else 'Models differ significantly (race is influential)'}")


---
## Module 08 Complete — Full Curriculum Summary

**NB-01** Reproducible foundations: project structure, requirements.txt, MANIFEST.json with SHA-256 checksums, pipeline logging with decorators, Makefile orchestration, reproducibility audit  
**NB-02** Quarto reports: YAML front matter, multi-format output (HTML/PDF/Word), publication figures, Table 1 with statistical tests, cross-references, parameterised reports  
**NB-03** Streamlit dashboards: multi-tab layout, KPI metrics, session state, caching, interactive filters, high-risk patient lists, download buttons, authentication patterns, Docker deployment  
**NB-04** MLflow tracking: experiments and runs, parameter/metric/artifact logging, model registry (staging/production/archived), drift detection visualisation, MLflow CLI  
**NB-05** FastAPI serving: Pydantic schemas with validation, single and batch prediction endpoints, health check, API testing suite, SHAP explain endpoint, Docker + docker-compose  
**NB-06** Testing & CI/CD: unittest for data pipeline and model performance, property-based testing (hypothesis), GitHub Actions workflow, pre-commit hooks, code quality metrics  
**NB-07** Model cards & governance: Mitchell et al. Model Card, fairness analysis by subgroup, EU AI Act/FDA context, governance checklist, ongoing monitoring, patient-facing explanation  
**NB-08** Capstone: full pipeline → run log → fairness → validation tests → FastAPI simulation → 300 dpi summary figure → deployment narrative → readiness checklist

**HealthPy Tutorial Series Complete: MOD-01 through MOD-08**
